# Interactive Agent Session: Chapter1_AutoGen_Financial

This Jupyter notebook provides a sandbox to run AI Agent code securely block-by-block. Make sure you have your `.env` configured properly before executing the agent loop below.

In [ ]:
import autogen

# Define the LLM configuration
llm_config = {"model": "gpt-4-turbo", "api_key": "YOUR_OPENAI_API_KEY"}

# 1. Create the User Proxy (Acts on behalf of the human and executes python code)
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    system_message="A human admin. Execute code written by the Analyst and report the output.",
    code_execution_config={"work_dir": "coding", "use_docker": False},
    human_input_mode="TERMINATE" 
)

# 2. Create the Financial Analyst Agent
financial_analyst = autogen.AssistantAgent(
    name="Financial_Analyst",
    system_message="You are a senior financial analyst. Write Python code to fetch stock data (using yfinance) and summarize the trends. You must fix any bugs in your code.",
    llm_config=llm_config,
)

# 3. Create the Risk Reviewer Agent
risk_reviewer = autogen.AssistantAgent(
    name="Risk_Reviewer",
    system_message="You are a Risk Manager. Review the financial report produced by the Analyst. Highlight any potential downside risks and add a 'Risk Score'. If the report lacks risk analysis, ask the Analyst to revise it.",
    llm_config=llm_config,
)

# Create a group chat to orchestrate the workflow
groupchat = autogen.GroupChat(
    agents=[user_proxy, financial_analyst, risk_reviewer], 
    messages=[], 
    max_round=10
)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

# Initiate the business process
user_proxy.initiate_chat(manager, message="Fetch the last 30 days of stock data for MSFT and AAPL. Provide a 2-paragraph comparative trend report and assess the investment risk.")
